### Behavioral Simulation Pipeline
This notebook demonstrates the complete Human Digital Twin (HDT) pipeline. It integrates emotion detection and behavioral decision prediction into a unified system.

User input text is processed to detect emotion, which is then mapped to a behavioral state. This mapped state, along with scenario and difficulty context, is used to predict a decision outcome. The system generates interpretable outputs including decision, confidence and explanation.

In [1]:
import numpy as np
import pandas as pd
import joblib

In [2]:
synthetic_model = joblib.load("../models/synthetic_model.pkl")
synthetic_encoder = joblib.load("../models/synthetic_encoder.pkl")
synthetic_scaler = joblib.load("../models/synthetic_scaler.pkl")
scenarios = joblib.load("../models/scenarios.pkl")
difficulty_map = joblib.load("../models/difficulty_map.pkl")
emotion_names = joblib.load("../models/goemotions_label_names.pkl")
goemotion_model = joblib.load("../models/goemotions_model.pkl")
vectorizer = joblib.load("../models/goemotions_vectorizer.pkl")
print("All model loaded successfully")

All model loaded successfully


In [3]:
emotion_mapping = {"nervousness" : "anxious", "fear" : "anxious", "sadness" : "stressed", "anger" : "stressed", "disappointment" : "stressed", "joy" : "thinking", "optimism" : "thinking", "curiosity" : "thinking", "confusion" : "thinking", "neutral" : "thinking"}

In [4]:
def simulate_behavior(user_text, scenario, difficulty):
    text_vector = vectorizer.transform([user_text])
    emotion_probs = goemotion_model.predict_proba(text_vector)[0]
    dominant_index = np.argmax(emotion_probs)
    dominant_emotion = (emotion_names[dominant_index])
    mapped_emotion = emotion_mapping.get(dominant_emotion, "thinking")
    synthetic_input = pd.DataFrame([{"scenario" : scenario, "difficulty" : difficulty, "emotion" : mapped_emotion, "response_time" : "slow" if difficulty in ["high", "very high"] else "moderate", "personality" : "balanced", "age_group" : "young", "stress_level" : "medium", "context" : "professional"}])
    encoded = synthetic_encoder.transform(synthetic_input)
    scaled = synthetic_scaler.transform(encoded)
    decision = synthetic_model.predict(scaled)[0]
    probabilities = synthetic_model.predict_proba(scaled)[0]
    confidence = np.max(probabilities)
    explanation = (f"The dominant detected emotion is '{dominant_emotion}', " f"which maps to the behavioral state '{mapped_emotion}'.\n" f"For the selected scenario '{scenario}' with difficulty level '{difficulty}', " f"the predicted decision tendency is '{decision}'.")
    return {"Predicted Decision" : decision, "Confidence" : round(float(confidence), 3), "Detected Emotion" : dominant_emotion, "Mapped Emotion" : mapped_emotion, "Explanation" : explanation}

In [5]:
def behavioral_interpretation(mapped_emotion, difficulty, decision):
    interpretation = []
    if mapped_emotion in ["anxious", "stressed", "panic", "overwhelmed"]:
        interpretation.append("Feeling anxious in challenging situations can increase the tendency to avoid decisions.")
    elif mapped_emotion in ["calm", "focused"]:
        interpretation.append("A calm emotional state usually supports clear and balanced decision-making.")
    elif mapped_emotion in ["concerned", "thinking"]:
        interpretation.append("A reflective emotional state may lead to careful and cautious decisions.")
    else:
        interpretation.append(f"The emotional state '{mapped_emotion}' influences how decisions are made under situational demands.")
    if difficulty in ["high", "very high"]:
        interpretation.append("Higher difficulty levels can increase emotional pressure and affect decision patterns.")
    elif difficulty == "medium":
        interpretation.append("Moderate difficulty allows better control over emotional reactions.")
    else:
        interpretation.append("Lower difficulty situations reduce emotional pressure on decisions.")
    if decision == "avoidant":
        interpretation.append("Using structured planning and breaking the problem into smaller steps may help reduce avoidance.")
    elif decision == "impulsive":
        interpretation.append("Taking a short pause before deciding may improve decision quality.")
    elif decision == "analytical":
        interpretation.append("Continuing structured analysis can support consistent decision outcomes.")
    elif decision == "cautious":
        interpretation.append("Balancing caution with timely action may improve results.")
    else:
        interpretation.append("Improving emotional awareness may enhance decision effectiveness.")
    return " ".join(interpretation)

In [6]:
user_text = input("Enter your situation: ")
print("\nAvailable Scenarios:")
print(scenarios)
scenario = input("Enter scenario from above list: ").strip().lower()
if scenario not in scenarios:
    print("Invalid scenario selected. Please choose a valid option from the list above.")
else:
    print(f"\nAvailable Difficulties for '{scenario}':")
    print(difficulty_map[scenario])
    difficulty = input("Enter difficulty from above list: ").strip().lower()
    if difficulty not in difficulty_map[scenario]:
        print("Invalid difficulty selected. Please choose a valid option for the selected scenario.")
    else:
        result = simulate_behavior(user_text = user_text, scenario = scenario, difficulty = difficulty)
        print("\n--- Behavioral Simulation Result ---\n")
        print(f"Predicted Decision : {result['Predicted Decision']}")
        print(f"Confidence Level   : {result['Confidence']}")
        print(f"Detected Emotion   : {result['Detected Emotion']}")
        print(f"Mapped Emotion     : {result['Mapped Emotion']}")
        print("\nExplanation:")
        print(result['Explanation'])
        print("\n--- Behavioral Interpretation ---\n")
        interpretation_text = behavioral_interpretation(mapped_emotion = result['Mapped Emotion'], difficulty = difficulty, decision = result['Predicted Decision'])
        print(interpretation_text)

Enter your situation:  I have a new job opportunity but i feel nervous and unsure.



Available Scenarios:
['career', 'money', 'ethics', 'relationship', 'health', 'risk', 'conflict', 'deadline']


Enter scenario from above list:  career



Available Difficulties for 'career':
['medium', 'high', 'very high']


Enter difficulty from above list:  high



--- Behavioral Simulation Result ---

Predicted Decision : avoidant
Confidence Level   : 1.0
Detected Emotion   : nervousness
Mapped Emotion     : anxious

Explanation:
The dominant detected emotion is 'nervousness', which maps to the behavioral state 'anxious'.
For the selected scenario 'career' with difficulty level 'high', the predicted decision tendency is 'avoidant'.

--- Behavioral Interpretation ---

Feeling anxious in challenging situations can increase the tendency to avoid decisions. Higher difficulty levels can increase emotional pressure and affect decision patterns. Using structured planning and breaking the problem into smaller steps may help reduce avoidance.
